In [1]:
import os
import gc
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AutoImageProcessor, ViTModel
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from tqdm import tqdm
import itertools
from PIL import Image

In [2]:


# ============================================
# DATA LOADING FUNCTIONS
# ============================================

def load_text_data(folder):
    """Load text files and return texts, labels, and filenames"""
    texts, labels, filenames = [], [], []
    
    # Check if folder exists
    if not os.path.exists(folder):
        print(f"Warning: {folder} does not exist")
        return [], np.array([]), []
    
    for label_name, label in [("Label_0", 0), ("Label_1", 1)]:
        subfolder = os.path.join(folder, label_name)
        
        if not os.path.exists(subfolder):
            print(f"Warning: {subfolder} does not exist")
            continue
            
        for file in sorted(os.listdir(subfolder)):
            if file.endswith(".txt"):
                with open(os.path.join(subfolder, file), "r", encoding="utf-8") as f:
                    texts.append(f.read())
                labels.append(label)
                filenames.append((label_name, file))
    
    return texts, np.array(labels), filenames


def load_image_paths(image_folder, filenames):
    """Load image paths corresponding to text files"""
    image_paths = []
    
    for label_name, file in filenames:
        img_name = file.replace(".txt", ".png")
        img_path = os.path.join(image_folder, label_name, img_name)
        
        if not os.path.exists(img_path):
            print(f"Warning: Missing image: {img_path}")
            # Try alternative extensions
            for ext in ['.jpg', '.jpeg']:
                alt_path = img_path.replace('.png', ext)
                if os.path.exists(alt_path):
                    img_path = alt_path
                    break
            else:
                raise FileNotFoundError(f"Missing image: {img_path}")
        
        image_paths.append(img_path)
    
    return image_paths

In [3]:



# ============================================
# FEATURE EXTRACTION
# ============================================

def extract_stylometric_features(code: str):
    """Extract 4 stylometric features"""
    lines = code.splitlines()
    avg_line_length = np.mean([len(line) for line in lines]) if lines else 0
    
    return np.array([
        avg_line_length,
        len(lines),
        len(code.split()),
        len(code)
    ], dtype=np.float32)


def compute_stylo(texts):
    """Compute stylometric features for all texts"""
    if len(texts) == 0:
        return np.array([]).reshape(0, 4)
    return np.array([extract_stylometric_features(t) for t in texts])

In [4]:



# ============================================
# DATASET CLASS
# ============================================

class MultiModalDataset(Dataset):
    def __init__(self, texts, images, labels, extra, tokenizer, processor, max_length=256):
        self.texts = texts
        self.images = images
        self.labels = labels
        self.extra = extra
        self.tokenizer = tokenizer
        self.processor = processor
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        # Load and process image
        img = Image.open(self.images[idx]).convert("RGB")
        
        # Tokenize text
        text_enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        
        # Process image
        img_enc = self.processor(images=img, return_tensors="pt")
        
        return {
            "input_ids": text_enc["input_ids"].squeeze(0),
            "attention_mask": text_enc["attention_mask"].squeeze(0),
            "pixel_values": img_enc["pixel_values"].squeeze(0),
            "extra": torch.tensor(self.extra[idx], dtype=torch.float32) if self.extra is not None and len(self.extra) > 0 else torch.zeros(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [5]:
# ============================================
# MODEL ARCHITECTURE
# ============================================

class HybridModel(nn.Module):
    def __init__(self, use_cb, use_vit, extra_dim):
        super().__init__()
        
        self.use_cb = use_cb
        self.use_vit = use_vit
        
        # Initialize models only if needed
        if use_cb:
            self.codebert = AutoModel.from_pretrained("microsoft/codebert-base")
            # Freeze CodeBERT
            # for param in self.codebert.parameters():
            #     param.requires_grad = False
        
        if use_vit:
            self.vit = ViTModel.from_pretrained("facebook/deit-base-patch16-224")
            # Freeze ViT
            # for param in self.vit.parameters():
            #     param.requires_grad = False
        
        # Calculate input dimension
        input_dim = 0
        if use_cb: 
            input_dim += 768
        if use_vit: 
            input_dim += 768
        input_dim += extra_dim
        
        self.norm = nn.LayerNorm(input_dim)
        
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 2)
        )
    
    def forward(self, batch):
        feats = []
        
        if self.use_cb:
            # with torch.no_grad():  # Freeze CodeBERT
            out = self.codebert(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"]
            )
            feats.append(out.last_hidden_state[:, 0, :])
        
        if self.use_vit:
            # with torch.no_grad():  # Freeze ViT
            out = self.vit(pixel_values=batch["pixel_values"])
            feats.append(out.last_hidden_state[:, 0, :])
        
        # Add extra features if they exist
        if batch["extra"].shape[1] > 0:
            feats.append(batch["extra"])
        
        # Concatenate all features
        x = torch.cat(feats, dim=1)
        x = self.norm(x)
        
        return self.classifier(x)

In [6]:
# ============================================
# TRAINING AND EVALUATION
# ============================================

def train_and_eval(combo, train_texts, train_labels, train_imgs, train_tfidf, train_metrics, 
                   vectorizer, scaler, device, tokenizer, processor):
    
    torch.cuda.empty_cache()
    gc.collect()
    
    use_cb = "codebert" in combo
    use_vit = "vit" in combo
    
    # Build extra features
    extra_list = []
    if "stylometric" in combo and len(train_texts) > 0:
        extra_list.append(compute_stylo(train_texts))
    if "metrics" in combo and train_metrics is not None:
        extra_list.append(train_metrics)
    if "tfidf" in combo and train_tfidf is not None:
        extra_list.append(train_tfidf)
    
    if extra_list:
        extra = np.concatenate(extra_list, axis=1)
        # Scale features
        scaler.fit(extra)
        extra = scaler.transform(extra)
    else:
        extra = np.zeros((len(train_texts), 0))
    
    # Create dataset and dataloader
    dataset = MultiModalDataset(train_texts, train_imgs, train_labels, extra, tokenizer, processor)
    loader = DataLoader(dataset, batch_size=4, shuffle=True)  # Reduced batch size for memory
    
    # Initialize model
    model = HybridModel(use_cb, use_vit, extra.shape[1]).to(device)
    
    # Only train the classifier head (feature extractors are frozen)
    optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
    loss_fn = nn.CrossEntropyLoss()
    
    # Training loop (3 epochs)
    model.train()
    for epoch in range(3):
        total_loss = 0
        for batch in loader:
            # Move batch to device
            batch = {k: v.to(device) for k, v in batch.items()}
            
            optimizer.zero_grad()
            outputs = model(batch)
            loss = loss_fn(outputs, batch["label"])
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        print(f"  Epoch {epoch+1}/3 - Loss: {total_loss/len(loader):.4f}")
    
    # Test on all 10 test sets
    accs = []
    
    for i in range(10):
        test_texts, test_labels, test_files = load_text_data(f"../text_files/Test_{i}")
        
        if len(test_texts) == 0:
            print(f"Test_{i}: No data found, skipping")
            continue
        
        test_imgs = load_image_paths(f"../snapshots/Test_{i}", test_files)
        
        # Build test extra features
        test_extra_list = []
        if "stylometric" in combo:
            test_extra_list.append(compute_stylo(test_texts))
        if "metrics" in combo:
            metrics_path = f"metrics_test_{i}.npz"
            if os.path.exists(metrics_path):
                test_extra_list.append(np.load(metrics_path)["test_metrics"])
            else:
                print(f"Warning: {metrics_path} not found")
        if "tfidf" in combo:
            test_extra_list.append(vectorizer.transform(test_texts).toarray())
        
        if test_extra_list:
            test_extra = np.concatenate(test_extra_list, axis=1)
            test_extra = scaler.transform(test_extra)
        else:
            test_extra = np.zeros((len(test_texts), 0))
        
        # Create test dataset
        test_dataset = MultiModalDataset(test_texts, test_imgs, test_labels, test_extra, tokenizer, processor)
        test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)
        
        # Evaluate
        model.eval()
        preds, ys = [], []
        
        with torch.no_grad():
            for batch in test_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                outputs = model(batch)
                preds.extend(outputs.argmax(dim=1).cpu().numpy())
                ys.extend(batch["label"].cpu().numpy())
        
        acc = accuracy_score(ys, preds)
        accs.append(acc)
        print(f"  Test_{i}: {acc:.4f}")
    
    # Cleanup
    del model, optimizer, dataset, loader
    torch.cuda.empty_cache()
    gc.collect()
    
    return np.mean(accs) if accs else 0.0

In [8]:
os.getcwd()

'/home/info-sec-lab/BTP/100k/experiments'

In [ ]:
# Load and display sizes of metrics arrays
metrics_train = np.load("../metrics/metrics_train.npz")
print("metrics_train.npz contents:")
for key in metrics_train.files:
    print(f"  {key}: shape {metrics_train[key].shape}")

metrics_test = np.load("../metrics/metrics_test.npz")
print("\nmetrics_test_0.npz contents:")
for key in metrics_test_0.files:
    print(f"  {key}: shape {metrics_test_0[key].shape}")

FileNotFoundError: [Errno 2] No such file or directory: 'metrics_train.npz'

In [14]:

# ============================================
# MAIN EXECUTION
# ============================================


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load tokenizer and processor
print("Loading tokenizer and processor...")
tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")
processor = AutoImageProcessor.from_pretrained("facebook/deit-base-patch16-224")

# Load training data
print("Loading training data...")
train_texts, train_labels, train_files = load_text_data("../text_files/train")

if len(train_texts) == 0:
    print("Error: No training data found!")
    exit(1)

print(f"Loaded {len(train_texts)} training samples")

# Load training images
train_imgs = load_image_paths("../snapshots/train", train_files)
print(f"Loaded {len(train_imgs)} training images")

# Compute TF-IDF features
print("Computing TF-IDF features...")
vectorizer = TfidfVectorizer(max_features=500)
train_tfidf = vectorizer.fit_transform(train_texts).toarray()

# Load pre-computed metrics (if available)
train_metrics = None
if os.path.exists("metrics_train.npz"):
    train_metrics = np.load("metrics_train.npz")["metrics"]
    print(f"Loaded metrics with shape: {train_metrics.shape}")
else:
    print("Warning: metrics_data.npz not found")
# train_metrics = train_metrics[:, 1:6]
# Initialize scaler
scaler = StandardScaler()

# Feature combinations to try
features = ["codebert", "vit", "stylometric", "tfidf", "metrics"]  # Removed 'metrics' if not available

best_acc = 0
best_combo = None

# Try all combinations
for r in range(1, len(features) + 1):
    for combo in itertools.combinations(features, r):
        print(f"\n{'='*50}")
        print(f"Testing combination: {combo}")
        print(f"{'='*50}")
        
        try:
            acc = train_and_eval(combo, train_texts, train_labels, train_imgs, 
                                train_tfidf, train_metrics, vectorizer, scaler, 
                                device, tokenizer, processor)
            print(f"\n>>> AVG Accuracy for {combo}: {acc:.4f} <<<\n")
            
            if acc > best_acc:
                best_acc = acc
                best_combo = combo
        except Exception as e:
            print(f"Error with combination {combo}: {e}")
            continue

print("\n" + "="*50)
print(f"BEST COMBINATION: {best_combo}")
print(f"BEST ACCURACY: {best_acc:.4f}")
print("="*50)


Using device: cuda
Loading tokenizer and processor...


Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


Loading training data...
Loaded 7520 training samples
Loaded 7520 training images
Computing TF-IDF features...
Loaded metrics with shape: (7520, 5)

Testing combination: ('codebert',)
  Epoch 1/3 - Loss: 0.6263
  Epoch 2/3 - Loss: 0.4625
  Epoch 3/3 - Loss: 0.3574
  Test_0: 0.5659
  Test_1: 0.6008
  Test_2: 0.6246
  Test_3: 0.5848
  Test_4: 0.6228
  Test_5: 0.5838
  Test_6: 0.6128
  Test_7: 0.6078
  Test_8: 0.6128
  Test_9: 0.6078

>>> AVG Accuracy for ('codebert',): 0.6024 <<<


Testing combination: ('vit',)


Some weights of ViTModel were not initialized from the model checkpoint at facebook/deit-base-patch16-224 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  Epoch 1/3 - Loss: 0.6451
  Epoch 2/3 - Loss: 0.5490
  Epoch 3/3 - Loss: 0.4736
  Test_0: 0.4910
  Test_1: 0.5240
  Test_2: 0.5094
  Test_3: 0.4990
  Test_4: 0.5230
  Test_5: 0.5180
  Test_6: 0.5230
  Test_7: 0.5130
  Test_8: 0.5230
  Test_9: 0.5130

>>> AVG Accuracy for ('vit',): 0.5136 <<<


Testing combination: ('stylometric',)
  Epoch 1/3 - Loss: 0.7056
  Epoch 2/3 - Loss: 0.7006
  Epoch 3/3 - Loss: 0.6977
  Test_0: 0.5719
  Test_1: 0.5070
  Test_2: 0.4374
  Test_3: 0.4261
  Test_4: 0.5010
  Test_5: 0.5579
  Test_6: 0.4541
  Test_7: 0.4551
  Test_8: 0.4541
  Test_9: 0.4551

>>> AVG Accuracy for ('stylometric',): 0.4820 <<<


Testing combination: ('tfidf',)
  Epoch 1/3 - Loss: 0.7012
  Epoch 2/3 - Loss: 0.6731
  Epoch 3/3 - Loss: 0.6502
  Test_0: 0.4760
  Test_1: 0.5269
  Test_2: 0.5744
  Test_3: 0.5529
  Test_4: 0.5279
  Test_5: 0.4850
  Test_6: 0.5739
  Test_7: 0.5539
  Test_8: 0.5739
  Test_9: 0.5539

>>> AVG Accuracy for ('tfidf',): 0.5399 <<<


Testing combination: ('metrics',

...

  Epoch 1/3 - Loss: 0.6525
  Epoch 2/3 - Loss: 0.5422
  Epoch 3/3 - Loss: 0.4641
Error with combination ('codebert', 'vit', 'tfidf', 'metrics'): 'test_metrics is not a file in the archive'

Testing combination: ('codebert', 'stylometric', 'tfidf', 'metrics')
  Epoch 1/3 - Loss: 0.6272
  Epoch 2/3 - Loss: 0.4387
  Epoch 3/3 - Loss: 0.3361
Error with combination ('codebert', 'stylometric', 'tfidf', 'metrics'): 'test_metrics is not a file in the archive'

Testing combination: ('vit', 'stylometric', 'tfidf', 'metrics')


Some weights of ViTModel were not initialized from the model checkpoint at facebook/deit-base-patch16-224 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  Epoch 1/3 - Loss: 0.6758
  Epoch 2/3 - Loss: 0.5614
  Epoch 3/3 - Loss: 0.4766
Error with combination ('vit', 'stylometric', 'tfidf', 'metrics'): 'test_metrics is not a file in the archive'

Testing combination: ('codebert', 'vit', 'stylometric', 'tfidf', 'metrics')


Some weights of ViTModel were not initialized from the model checkpoint at facebook/deit-base-patch16-224 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  Epoch 1/3 - Loss: 0.6384
  Epoch 2/3 - Loss: 0.4533
  Epoch 3/3 - Loss: 0.3252
Error with combination ('codebert', 'vit', 'stylometric', 'tfidf', 'metrics'): 'test_metrics is not a file in the archive'

BEST COMBINATION: ('codebert', 'stylometric')
BEST ACCURACY: 0.6480
